# PlanTo3D — upload a floor plan, get a house

Upload any plan, as a PDF or an image. Run the cells in order.

**Use the 512 checkpoint.** The 768 one trained longer and is not better:
window IoU rose 0.096 to 0.113 while room and bath fell, and downstream it
found fewer openings and calibrated fewer plans correctly. Section 2
prints which one is loaded.

## Where it stands

Every figure below was produced by the named script over 60 plans from
CubiCasa's held-out test split, which the model never trained on (listed in
`data/cubicasa_test60.txt`). `docs/AUDIT.md` records how.

| | Measured | Script |
| --- | --- | --- |
| **Plans right on every check at once** | **27 of 60** | `output_scorecard.py` |
| Plans that fail to build a model | 0 of 60 | `output_scorecard.py` |
| Plans with the wrong number of storeys | 3 of 60 | `output_scorecard.py` |
| Open-to-sky spaces, pixel IoU | **75.4%** over 52 single-plan sheets | `open_air_accuracy.py` |
| Scale error, median | 12.9% (doors 8.7%, walls 16.7%) | `scale_accuracy.py` |

The most visible recent fix: on some sheets a green brand colour on the
walls was read as planting, closed into one region covering the whole
plan, and the roof above it was removed. A planted region may now cover at
most half the sheet, which removes that failure without touching real
gardens.

## What is still weak

**Scale** is the commonest failure, 16 of 60. When a sheet prints room
sizes they are read and checked against the geometry; otherwise size comes
from door widths (good) or wall thickness (about a fifth too small). Section
5 says which one this drawing used.

**Windows** are the weakest detection, and the evidence points to the
training data, where windows are about 0.1% of annotated pixels.

**Room names** are rarely readable on scanned sheets. When the model gets a
room wrong, `scripts/correct_and_build.py` relabels it and rebuilds.

## 1. Setup

Dependencies, then the code. Two or three minutes.

In [ ]:
!apt-get -qq install -y poppler-utils tesseract-ocr > /dev/null
!pip install -q trimesh shapely mapbox-earcut pytesseract pdf2image \
    segmentation-models-pytorch diffusers transformers accelerate safetensors
print("dependencies ready")

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/priyanshsoni096-blip/PlanTo3D.git"
repo = Path("/content/PlanTo3D")

# Fetched and reset rather than pulled: a pull merges, and merging fails
# against a rewritten history. There is nothing in this clone worth keeping.
if repo.exists():
    !cd {repo} && git fetch --quiet origin && git reset --hard --quiet origin/main && git clean -qfd
else:
    !git clone --quiet {REPO_URL} {repo}

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

# Colab keeps modules loaded between runs, so new code on disk changes
# nothing that is running until these are dropped.
for name in [m for m in sys.modules if m.startswith(("planto3d", "training"))]:
    del sys.modules[name]

import torch

!cd {repo} && git log --oneline -1
print(
    f"\nGPU: {torch.cuda.get_device_name(0)}"
    if torch.cuda.is_available()
    else "\nNo GPU -- everything works except the photoreal pass in section 8."
)

## 2. The trained model

The checkpoint is ~98 MB and is not in the repository. Keep it on Drive at
`MyDrive/planto3d/unet_cubicasa.pt` and this finds it; otherwise upload it
when prompted.

Without one the classical baseline runs instead, which only reads cleanly
drafted CAD sheets and will not know what any room is for.

In [ ]:
from google.colab import drive, files

models = repo / "models"
models.mkdir(exist_ok=True)

drive.mount("/content/drive")
saved = Path("/content/drive/MyDrive/planto3d/unet_cubicasa.pt")

if saved.is_file():
    !cp "{saved}" "{models}/unet_cubicasa.pt"
    print(f"using {saved}")
else:
    print("No checkpoint on Drive. Upload unet_cubicasa.pt below.")
    for name in files.upload():
        Path(name).rename(models / name)

from planto3d.segment import load_segmenter

CHECKPOINT = next(iter(sorted(models.glob("*.pt"))), None)
SEGMENTER = load_segmenter(CHECKPOINT)

if CHECKPOINT:
    state = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
    print(
        f"\n{state.get('num_classes')} classes, trained to epoch "
        f"{state.get('epoch')}, validation Dice {float(state.get('val_dice', 0)):.4f}"
    )
    if state.get("num_classes", 5) < 11:
        print("This checkpoint predates room types -- rooms will come out plain.")

## 3. Your floor plan

**PDF, PNG or JPEG.** Upload the original rather than a screenshot: room
names drive some of the detail and reading them needs resolution.

One file per storey, ground floor first, or a single multi-page PDF.

In [ ]:
uploaded = sorted(files.upload())
plans = Path("/content/plan")
!rm -rf {plans}
plans.mkdir(parents=True)

for index, name in enumerate(uploaded):
    Path(name).rename(plans / f"{index:02d}{Path(name).suffix.lower()}")

SOURCE = plans if len(uploaded) > 1 else next(plans.iterdir())
print(f"{len(uploaded)} file(s) -> {SOURCE}")

## 4. How it should look

Five choices, and none of them can produce an ugly result. Re-run this cell
after changing any of them, then re-run section 5.

**Style** is what the building is clad in. **Colour** takes that lighter,
darker or warmer. **Time** is the hour it is seen at. **Landscaping**
decides how much of a setting it gets -- `none` leaves it alone against the
sky, which is what a massing study wants and a presentation render does
not. **Creativity** only affects section 8: strict holds the geometry and
looks increasingly like a shaded model, creative invents freely and stops
describing this particular building.

In [ ]:
#@title Choose, then run { run: "auto" }
style = "luxury"  #@param ["modern", "luxury", "traditional", "minimalist"]
colour = "warm"  #@param ["light", "dark", "warm"]
time_of_day = "day"  #@param ["day", "sunset", "night"]
landscaping = "premium"  #@param ["none", "basic", "premium"]
creativity = "balanced"  #@param ["strict", "balanced", "creative"]
storey_height_ft = 9  #@param {type:"slider", min:7, max:14, step:0.5}

from planto3d.design import Design

DESIGN = Design(
    style=style,
    colour=colour,
    time=time_of_day,
    landscaping=landscaping,
    creativity=creativity,
)
print(DESIGN)

## 5. Read the drawing and build it

A minute or two. The readout afterwards is worth reading rather than
scrolling past -- it says how confident the result is and why.

In [ ]:
import logging

from planto3d.pipeline import run

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s")

OUT = Path("/content/output")
!rm -rf {OUT}

result = run(
    SOURCE,
    OUT,
    segmenter=SEGMENTER,
    wall_height_ft=storey_height_ft,
    palette=DESIGN.palette(),
    site=DESIGN.site(),
)

print(f"{len(result.floors)} storey(s)")
print(f"{result.wall_count} walls, {result.room_count} rooms, {result.opening_count} openings")

SOURCES = {
    "dimensions": "measured from the printed room dimensions",
    "areas": "measured from a printed floor area",
    "stated": "stated by you",
    "doors": "inferred from door widths, taking a standard 2'6\" door",
    "walls": "inferred from wall thickness, taking a standard 9\" wall",
    "ratio": "assumed from a 1:150 drafting ratio -- nothing measurable was found",
}
print(f"\nScale: {result.scale:.1f} px/ft, {SOURCES.get(result.scale_source, result.scale_source)}")
if result.scale_assumed:
    print("  The proportions are right; the absolute size is an estimate.")

print()
for floor in result.floors:
    types = sorted({r.category for r in floor.plan.rooms if r.category})
    names = ", ".join(floor.named_rooms) or "none read"
    print(f"floor {floor.index + 1}: {len(floor.plan.walls)} walls, {len(floor.plan.rooms)} rooms")
    print(f"   room types : {', '.join(types) if types else 'none predicted'}")
    print(f"   names read : {names}")

named = sum(len(f.named_rooms) for f in result.floors)
typed = sum(1 for f in result.floors for r in f.plan.rooms if r.category)
if not typed and not named:
    print("\nNothing is known about what any room is for, so the model will be")
    print("plain. Usually an input resolution problem -- upload the original PDF.")

## 6. What was actually detected

**The most useful picture here.** Red is walls, green is rooms, over your
own drawing. If the model looks wrong, this says whether the drawing was
misread or the geometry mishandled it -- and those need different fixes.

In [ ]:
import cv2
import matplotlib.pyplot as plt

from planto3d.pipeline import draw_overlay

figure, axes = plt.subplots(
    1, len(result.floors), figsize=(11 * len(result.floors), 11), squeeze=False
)
for axis, floor in zip(axes[0], result.floors):
    axis.imshow(cv2.cvtColor(draw_overlay(floor), cv2.COLOR_BGR2RGB))
    axis.set_title(f"floor {floor.index + 1}: walls red, rooms green", fontsize=13)
    axis.axis("off")
plt.tight_layout()
plt.show()

## 7. The house &mdash; this is the deliverable

Six views of the model built from your drawing. **Everything here traces
back to a line on the sheet**, and every part of it has been scored
against ground truth: `docs/AUDIT.md` names the script behind each figure.

Judge it on fidelity. Check the elevations against your drawing wall by
wall, and count the storeys and rooms. A blank elevation usually means
that wall genuinely has no openings rather than that something failed.

Where it currently stands, over 60 held-out plans scored end to end: 27
come out right on every count at once. Size is the commonest failure, at 16 of 60,
and section 5 tells you whether *this* drawing gave consistent evidence
about its own size.

In [ ]:
from PIL import Image

from planto3d.preview import render_views

views = render_views(
    result.model_path, OUT, resolution=(1100, 800), lighting=DESIGN.lighting()
)

order = ["aerial", "front", "back", "left", "right", "top"]
figure, axes = plt.subplots(2, 3, figsize=(22, 11))
for axis, name in zip(axes.ravel(), order):
    axis.imshow(Image.open(views[name]))
    axis.set_title(name, fontsize=14)
    axis.axis("off")
plt.tight_layout()
plt.show()

## 7b. The same house, path-traced &mdash; optional

The same measured model rendered with Blender Cycles: real materials, soft
shadows and a sky for the hour chosen in section 4. **Nothing is invented**
&mdash; unlike section 8, every surface here is one the drawing supports.

Blender's Python module only installs on Python 3.13. This cell checks the
runtime first and skips cleanly if it cannot run. On a CPU runtime expect a
few minutes for all six views.

In [ ]:
import sys

if sys.version_info[:2] != (3, 13):
    print(f"Skipped: Blender's bpy needs Python 3.13, this runtime is "
          f"{sys.version_info.major}.{sys.version_info.minor}. Section 7 is the result.")
else:
    !pip install -q "bpy>=5.2,<6"
    from planto3d import blender_render

    if not blender_render.available():
        print("Skipped: bpy did not import on this runtime.")
    else:
        rendered = blender_render.render_views(
            result.model_path, OUT / "blender",
            resolution=(1100, 825), lighting=DESIGN.lighting(),
        )
        figure, axes = plt.subplots(2, 3, figsize=(22, 11))
        for axis, name in zip(axes.ravel(), ["aerial", "front", "back", "left", "right", "top"]):
            axis.imshow(Image.open(rendered[name]))
            axis.set_title(f"{name} (Cycles)", fontsize=14)
            axis.axis("off")
        plt.tight_layout()
        plt.show()

## 8. Photoreal &mdash; an impression, not a result

**Nothing below is measured, and it is not the output of the pipeline.**

This is Stable Diffusion, conditioned on the model's depth so it dresses
*this* building rather than inventing a different one. But it invents by
construction: the stone coursing, the dusk light, the reflections and the
planting are not on your drawing and were never read from it. It has
never been scored against anything, because there is nothing to score it
against &mdash; there is no ground truth for what a house should look like.

Two things follow, and they cut both ways:

- A beautiful image here does **not** mean the geometry is right. It will
  look convincing over a model with the wrong number of storeys.
- An awkward image does **not** mean the geometry is wrong. Diffusion is
  perfectly capable of making a correct model look strange.

So judge section 7 for whether the building is right, and this for whether
you like it. Do not let either answer stand in for the other.

Needs the GPU, and the first run downloads about 4 GB. The prompt is built
from what the pipeline actually read &mdash; cars where it found parking, lawn
where it found planting &mdash; and how far it may stray comes from the
creativity setting in section 4.

In [ ]:
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler,
)

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4, then re-run.")

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11f1p_sd15_depth", torch_dtype=torch.float16
)
PIPE = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
)
PIPE.scheduler = UniPCMultistepScheduler.from_config(PIPE.scheduler.config)
PIPE = PIPE.to("cuda")
PIPE.enable_attention_slicing()
print("ready")

In [ ]:
from planto3d.photoreal import build_guides, build_negative_prompt, build_prompt

guides = build_guides(result.model_path, OUT / "guides")
depth = Image.open(guides["depth"]).convert("RGB")

# SD 1.5 is happiest near 768 and works in multiples of eight.
longest = max(depth.size)
if longest > 768:
    depth = depth.resize(tuple(int(d * 768 / longest) for d in depth.size))
width, height = (max(d - d % 8, 8) for d in depth.size)

labels = [label for floor in result.floors for label in floor.named_rooms]
negative_prompt = build_negative_prompt(DESIGN)
prompt = build_prompt(len(result.floors), labels, design=DESIGN)
print(f"{width}x{height}, conditioning {DESIGN.conditioning()}\n\n{prompt}\n")

image = PIPE(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=depth.resize((width, height)),
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=DESIGN.conditioning(),
    generator=torch.Generator(device="cuda").manual_seed(7),
).images[0]

image.save(OUT / "photoreal.png")
plt.figure(figsize=(14, 11))
plt.imshow(image)
plt.axis("off")
plt.show()

## 9. Take it home

A zip of everything. Two kinds of thing are in it and they are not
interchangeable:

| | What it is |
| --- | --- |
| `house.glb`, the six views, the overlays | the **result** &mdash; measured, and traceable to your drawing |
| `photoreal.png` | an **impression** &mdash; unmeasured, and partly invented |

The `.glb` opens in Windows 3D Viewer, Blender, or anything that reads
glTF.

In [ ]:
import shutil

for floor in result.floors:
    cv2.imwrite(str(OUT / f"detected-{floor.index + 1}.png"), draw_overlay(floor))

shutil.make_archive("/content/planto3d_output", "zip", OUT)
print(f"{sum(1 for _ in OUT.rglob('*') if _.is_file())} files")
files.download("/content/planto3d_output.zip")